In [102]:
from pathlib import Path
import sys
from typing import Any, TypedDict

import pandas as pd
from IPython.display import display
from langgraph.graph import StateGraph, START, END

# Supports running from the repository root or its notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "processed" / "songs_cleaned.csv").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "songs_cleaned.csv"
MODEL_DIR = PROJECT_ROOT / "models"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Missing dataset: {DATA_PATH}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.recommender import SongRecommender

songs = pd.read_csv(DATA_PATH)
required_columns = {"track_id", "track_name", "artists", "album_name"}
missing_columns = required_columns - set(songs.columns)
if missing_columns:
    raise ValueError(f"Missing dataset columns: {sorted(missing_columns)}")
if songs["track_id"].isna().any() or songs["track_id"].duplicated().any():
    raise ValueError("Track IDs must be present and unique")

print("Dataset shape:", songs.shape)
print("Unique track IDs:", songs["track_id"].nunique())
display(songs[["track_id", "track_name", "artists", "album_name"]].head())

Dataset shape: (34466, 19)
Unique track IDs: 34466


,track_id,track_name,artists,album_name
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,Comedy
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,Ghost (Acoustic)
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,Hold On


In [103]:
def find_song(song_name: str, catalog: pd.DataFrame) -> pd.DataFrame:
    """Return all exact, case-insensitive title matches."""
    query = song_name.strip().casefold()
    normalized_titles = catalog["track_name"].fillna("").astype(str).str.strip().str.casefold()
    columns = ["track_id", "track_name", "artists", "album_name"]
    return catalog.loc[normalized_titles == query, columns].copy()

assert find_song("ThisSongDoesNotExist123", songs).empty
assert set(find_song("COMEDY", songs)["track_id"]) == set(find_song("comedy", songs)["track_id"])
assert "album_name" in find_song("Comedy", songs).columns

for title in ["Comedy", "Blinding Lights", "Shape of You", "ThisSongDoesNotExist123"]:
    print(f"{title}: {len(find_song(title, songs))} match(es)")
print("Lookup checks passed.")

Comedy: 1 match(es)
Blinding Lights: 12 match(es)
Shape of You: 1 match(es)
ThisSongDoesNotExist123: 0 match(es)
Lookup checks passed.


In [104]:
recommender = SongRecommender(data_path=DATA_PATH, model_dir=MODEL_DIR)
seed_track_id = "5SuOikwiRyPMVoIQDJUgSV"  # Comedy — Gen Hoshino
sample_recommendations = recommender.recommend(track_id=seed_track_id, top_n=10)
expected_columns = {"track_id", "track_name", "artists", "album_name", "similarity_score"}
assert expected_columns.issubset(sample_recommendations.columns)
assert 0 < len(sample_recommendations) <= 10
assert seed_track_id not in sample_recommendations["track_id"].astype(str).tolist()
print("Standalone model output:", sample_recommendations.shape)
display(sample_recommendations.head(10))

Standalone model output: (10, 5)


,track_id,track_name,artists,album_name,similarity_score
0,3IORLAm07yxo4T7B8cRET3,Look For The Good (Single Version),Jason Mraz,Look For The Good,0.934962
1,54XZSt4qoPwUh0s9q0A7oA,Blink 182,Trille,Blink 182,0.934423
2,6hDBkm6B8HF9B4oATW28YN,Pop Virus,Gen Hoshino,POP VIRUS,0.918056
3,7cP99Dd9VNjDpr4yYIRGK9,Sonhos Gangsta,Flora Matos,Eletrocardiograma,0.911466
4,04V3hO138BCddFy9TmTUic,JAMAICA,Feid;Sech,Halloween 2022 Perreo Vol. 4,0.905246
5,2puCdLnS6sKRdbEabw4bjP,Bad Ass Bitches,Wiz Khalifa,Ghetto Superstar - New Rap Classics,0.885024
6,3yzkKgMeqMQjZl5BU40qC7,Severlá - Otto y los Sarcófagos del Ritmo,31 Minutos,31 Canciones de Amor y una Canción de Guaripolo,0.876092
7,13SgfMDRRldjdxzd6eKpQm,Amigos,Juan Magán;Mariah Angeliq;Yera,Hora del taco sin auto,0.872186
8,4gbx3Zyi7k4UOZWd6FU6aD,"Hard Talk, Pt. 2",TJ Cream;Marko,Wisdom Seeker,0.864196
9,23tKvchhAzfCJW8HkBlxf6,Vacilón,Don Omar,Fiesta Reggaeton 2010s,0.864175


In [105]:
class SongState(TypedDict):
    user_input: str
    song: str
    requested_artist:str 
    track_id: str
    artist: str
    album: str
    status: str
    response: str
    recommendations: list[dict[str, Any]]


In [106]:


def make_initial_state(user_input: str) -> SongState:
    """Create a fresh state for each request."""
    return {
        "user_input": user_input,
        "song": "",
        "requested_artist":"",
        "track_id": "",
        "requested_artist": "",
        "album": "",
        "status": "",
        "response": "",
        "recommendations": [],
    }

In [114]:
def identify_song(state: SongState) -> dict:
    """Extract the song title and optional artist."""

    user_input = state["user_input"].strip()
    prefix = "Recommend songs like "

    # Validate request format.
    if not user_input.casefold().startswith(prefix.casefold()):
        return {
            "status": "invalid_input",
            "response": "Use: Recommend songs like <song title> [by <artist>]"
        }

    # Extract the text after the prefix.
    query = user_input[len(prefix):].strip()

    if not query:
        return {
            "status": "invalid_input",
            "response": "Please provide a song title."
        }

    # Extract the optional artist.
    separator_position = query.casefold().rfind(" by ")

    if separator_position != -1:
        song = query[:separator_position].strip()
        artist = query[separator_position + 4:].strip()
    else:
        song = query
        artist = ""

    if not song or (separator_position != -1 and not artist):
        return {
            "status": "invalid_input",
            "response": "Provide a valid song title and artist."
        }

    return {
        "song": song,
        "requested_artist": artist,
        "status": "pending",
        "response": ""
    }

In [108]:
def find_song(
    song_name: str,
    songs: pd.DataFrame,
    artist_name: str = ""
) -> pd.DataFrame:
    """Find songs by exact title and optionally by artist."""

    title_query = song_name.strip().casefold()

    normalized_titles = (
        songs["track_name"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
    )

    matches = songs.loc[
        normalized_titles == title_query,
        ["track_id", "track_name", "artists", "album_name"]
    ].copy()

    if artist_name.strip():
        artist_query = artist_name.strip().casefold()

        # Artists in this dataset can be separated by semicolons.
        artist_matches = matches["artists"].fillna("").astype(str).apply(
            lambda value: artist_query in [
                name.strip().casefold()
                for name in value.split(";")
            ]
        )

        matches = matches.loc[artist_matches].copy()

    return matches

In [109]:
def lookup_song(state: SongState) -> dict:
    """Resolve a track using its title and optional artist."""

    matches = find_song(
        song_name=state["song"],
        songs=songs,
        artist_name= state["requested_artist"]
    )

    if matches.empty:
        return {
            "status": "not_found",
            "response": (
                f"No matching track found for '{state['song']}'"
                + (
                    f" by {state['requested_artist']}."
                    if state["requested_artist"]
                    else "."
                )
            )
        }

    if len(matches) > 1:
        options = matches[
            ["track_name", "artists", "album_name"]
        ].drop_duplicates().head(10)

        descriptions = [
            f"{row.track_name} — {row.artists} "
            f"(Album: {row.album_name})"
            for row in options.itertuples(index=False)
        ]

        return {
            "status": "ambiguous",
            "response": (
                f"Found {len(matches)} matching tracks. "
                "Please clarify the artist or album. "
                "Options: " + "; ".join(descriptions)
            )
        }

    selected = matches.iloc[0]

    return {
        "track_id": str(selected["track_id"]),
        "artist": str(selected["artists"]),
        "album": str(selected["album_name"]),
        "status": "found",
        "response": (
            f"Found '{selected['track_name']}' "
            f"by {selected['artists']} "
            f"from the album '{selected['album_name']}'."
        )
    }

In [110]:
def recommend_songs(state: SongState) -> dict:
    """Call the existing recommender for a uniquely identified song."""
    results = recommender.recommend(track_id=state["track_id"], top_n=10)
    records = results.to_dict(orient="records")
    return {
        "recommendations": records,
        "status": "recommended",
        "response": (
            f"Generated {len(records)} recommendations for "
            f"'{state['song']}' by {state['artist']}."
        ),
    }

In [111]:
def route_after_identification(state: SongState) -> str:
    return "stop" if state["status"] == "invalid_input" else "lookup"


def route_after_lookup(state: SongState) -> str:
    return "recommend" if state["status"] == "found" else "stop"


builder = StateGraph(SongState)
builder.add_node("identify_song", identify_song)
builder.add_node("lookup_song", lookup_song)
builder.add_node("recommend_songs", recommend_songs)
builder.add_edge(START, "identify_song")
builder.add_conditional_edges(
    "identify_song", route_after_identification,
    {"lookup": "lookup_song", "stop": END},
)
builder.add_conditional_edges(
    "lookup_song", route_after_lookup,
    {"recommend": "recommend_songs", "stop": END},
)
builder.add_edge("recommend_songs", END)
graph = builder.compile()
print("Three-node LangGraph compiled.")

Three-node LangGraph compiled.


In [112]:
test_inputs = [
    "Recommend songs like Rock",
    "Recommend songs like Numb",
    "Recommend songs like Numb by Linkin Park"
]

for user_input in test_inputs:
    result = graph.invoke(make_initial_state(user_input))

    print("=" * 60)
    print("Input:", user_input)
    print("Parsed song:", repr(result["song"]))
    print("Requested artist:", repr(result["requested_artist"]))
    print("Status:", result["status"])
    print("Response:", result["response"])

Input: Recommend songs like Rock
Parsed song: 'Rock'
Requested artist: ''
Status: recommended
Response: Generated 10 recommendations for 'Rock' by Kapanga.
Input: Recommend songs like Numb
Parsed song: 'Numb'
Requested artist: ''
Status: ambiguous
Response: Found 6 matching tracks. Please clarify the artist or album. Options: Numb — Marshmello;Khalid (Album: Numb); Numb — Arc North;Aaron Richards;New Beat Order;Cour (Album: Numb); Numb — Linkin Park (Album: Meteora); Numb — Linkin Park (Album: Meteora (Bonus Edition)); Numb — Khan Bhaini (Album: Numb); Numb — Lithe (Album: Numb)
Input: Recommend songs like Numb by Linkin Park
Parsed song: 'Numb'
Requested artist: 'Linkin Park'
Status: ambiguous
Response: Found 2 matching tracks. Please clarify the artist or album. Options: Numb — Linkin Park (Album: Meteora); Numb — Linkin Park (Album: Meteora (Bonus Edition))


In [113]:
test_cases = {
    "Recommend songs like Comedy": "recommended",
    "Recommend songs like Blinding Lights": "ambiguous",
    "Recommend songs like ThisSongDoesNotExist123": "not_found",
    "Hello": "invalid_input",
    "Recommend songs like ": "invalid_input",
}

for user_input, expected_status in test_cases.items():
    result = graph.invoke(make_initial_state(user_input))
    assert result["status"] == expected_status, (user_input, result["status"])
    if expected_status == "recommended":
        records = result["recommendations"]
        assert len(records) == 10
        assert len({item["track_id"] for item in records}) == len(records)
        assert all(item["track_id"] != result["track_id"] for item in records)
        assert result["album"]
    else:
        assert result["recommendations"] == []
    print(f"{result['status']:15} | {len(result['recommendations']):2} recommendations | {user_input}")

print("All integration tests passed.")

recommended     | 10 recommendations | Recommend songs like Comedy
ambiguous       |  0 recommendations | Recommend songs like Blinding Lights


not_found       |  0 recommendations | Recommend songs like ThisSongDoesNotExist123
invalid_input   |  0 recommendations | Hello
invalid_input   |  0 recommendations | Recommend songs like 
All integration tests passed.
